# NeuralMath — neural network from scratch

Companion notebook for **NeuralMath.org** and the pedagogical text *A anatomia matemática de uma rede neural*. This notebook reproduces the current article example: 2 inputs (mass and diameter), **12 ReLU hidden neurons**, one sigmoid output, **64 synthetic training examples**, 8 illustrative test examples, quadratic loss, explicit backpropagation, and stochastic gradient descent.

The code is intentionally explicit so that each mathematical operation can be inspected and related to the equations.

## 1. Model ingredients

For one standardized input vector $\mathbf{x}$, the network computes

$$
\mathbf{z}^{(1)}=W^{(1)}\mathbf{x}+\mathbf{b}^{(1)},\qquad
\mathbf{h}=\operatorname{ReLU}(\mathbf{z}^{(1)}),
$$

$$
z^{(2)}=W^{(2)}\mathbf{h}+b^{(2)},\qquad
\widehat y=\sigma(z^{(2)}).
$$

We implement the activation functions and their derivatives directly; no automatic differentiation is used.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def relu(s):
    return np.maximum(0, s)


def relu_derivative(s):
    return (s > 0).astype(float)


def sigmoid(s):
    return 1 / (1 + np.exp(-s))


def base_boundary(m):
    return (
        6.68
        + 0.74 * np.exp(-((m - 155) / 20) ** 2)
        + 1.50 / (1 + np.exp(-(m - 202) / 5))
    )


## 2. Synthetic data and standardization

The data are pedagogical but remain in plausible mass–diameter ranges. The network never receives the reference curve used to generate the geometry. Means and standard deviations are computed from the training set only.

In [ ]:
def build_dataset():
    masses_1 = np.linspace(118, 212, 16)
    masses_2 = masses_1 + np.array([
        1.2, -1.1, 0.8, -1.3, 1.0, -0.8, 1.1, -1.0,
        1.3, -0.9, 0.9, -1.2, 1.0, -1.0, 0.8, -0.7,
    ])
    k = np.arange(16)

    apple_1 = np.column_stack([
        masses_1,
        base_boundary(masses_1) + 0.22 + 0.05 * np.sin(0.8 * k),
    ])
    apple_2 = np.column_stack([
        masses_2,
        base_boundary(masses_2) + 0.58 + 0.06 * np.cos(0.45 * k),
    ])
    orange_1 = np.column_stack([
        masses_1,
        base_boundary(masses_1) - (0.22 + 0.04 * np.cos(0.7 * k)),
    ])
    orange_2 = np.column_stack([
        masses_2,
        base_boundary(masses_2) - (0.55 + 0.05 * np.sin(0.5 * k)),
    ])

    X_train = np.vstack([apple_1, apple_2, orange_1, orange_2])
    y_train = np.array([1.0] * 32 + [0.0] * 32)

    X_test = np.array([
        [121.0, 7.20], [135.0, 7.45], [152.0, 7.75], [171.0, 7.32],
        [121.0, 6.28], [152.0, 6.86], [189.0, 6.48], [208.0, 7.55],
    ])
    y_test = np.array([1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0])
    return X_train, y_train, X_test, y_test


X_train, y_train, X_test, y_test = build_dataset()
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
Xn = (X_train - mean) / std

print('Training set' + ':', X_train.shape)
print('Test set' + ':', X_test.shape)
print('Mean' + ':', mean)
print('Std' + ':', std)


## 3. Training with explicit backpropagation

For the quadratic loss $E=\tfrac12(y-\widehat y)^2$, every derivative is written explicitly. The parameters are updated after each shuffled training example.

In [ ]:
rng = np.random.default_rng(42)
W1 = 0.1 * rng.standard_normal((12, 2))
b1 = np.zeros(12)
W2 = 0.1 * rng.standard_normal(12)
b2 = 0.0

learning_rate = 0.05
n_epochs = 6000
history = []

for epoch in range(n_epochs):
    for i in rng.permutation(len(Xn)):
        x = Xn[i]
        target = y_train[i]

        # Forward pass
        z1 = W1 @ x + b1
        h = relu(z1)
        z2 = W2 @ h + b2
        y_hat = sigmoid(z2)

        # Backpropagation
        delta2 = (y_hat - target) * y_hat * (1 - y_hat)
        dW2 = delta2 * h
        db2 = delta2

        delta1 = (delta2 * W2) * relu_derivative(z1)
        dW1 = np.outer(delta1, x)
        db1 = delta1

        # Stochastic-gradient update
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

    errors = []
    for x, target in zip(Xn, y_train):
        h = relu(W1 @ x + b1)
        y_hat = sigmoid(W2 @ h + b2)
        errors.append(0.5 * (target - y_hat) ** 2)
    history.append(float(np.mean(errors)))

print(f"Final mean loss: {history[-1]:.10f}")


## 4. Predictions and learned decision boundary

After training, $\widehat y\ge 0.5$ is classified as an apple and $\widehat y<0.5$ as an orange. The level $\widehat y=0.5$ is the learned decision boundary.

In [ ]:
def predict(X):
    X_scaled = (X - mean) / std
    outputs = []
    for x in X_scaled:
        h = relu(W1 @ x + b1)
        outputs.append(sigmoid(W2 @ h + b2))
    return np.array(outputs)


pred_train = predict(X_train)
pred_test = predict(X_test)

print(f"Training accuracy: {np.mean((pred_train >= 0.5) == y_train):.4f}")
print(f"Illustrative test accuracy: {np.mean((pred_test >= 0.5) == y_test):.4f}")
print('Test outputs' + ':', np.round(pred_test, 6))

masses = np.linspace(116, 214, 500)
diameters = np.linspace(6.0, 8.85, 500)
M, D = np.meshgrid(masses, diameters)
grid = np.column_stack([M.ravel(), D.ravel()])
P = predict(grid).reshape(M.shape)

fig, ax = plt.subplots(figsize=(9.2, 6.3))
ax.contourf(
    M, D, P,
    levels=[0.0, 0.25, 0.50, 0.75, 1.0],
    colors=['#f3e0cb', '#ecd5c4', '#dbe7cf', '#cde3bf'],
    alpha=0.95,
)
ax.contour(M, D, P, levels=[0.5], colors=['#5a1a8a'], linewidths=2.2)

mask = y_train == 1.0
ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='o', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='Apple — train')
mask = y_train == 0.0
ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='s', s=48,
           facecolors='white', edgecolors='black', linewidths=1.1,
           label='Orange — train')
mask = y_test == 1.0
ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='^', s=92,
           color='#1f77b4', label='Apple — test')
mask = y_test == 0.0
ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='x', s=110,
           color='#ff7f0e', linewidths=2.0, label='Orange — test')

ax.set_xlim(116, 214)
ax.set_ylim(6.0, 8.85)
ax.set_xlabel('Mass (g)')
ax.set_ylabel('Diameter (cm)')
ax.set_title('Neural network output after training', fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=True)
fig.tight_layout()
plt.show()


## 5. What to observe

The final plot connects the algebra to the geometry. The shaded regions show the classes predicted by the network, while the highlighted curve is the learned level $\widehat y=0.5$. Because the training data are synthetic and constructed for teaching, the reported accuracy is an illustration of the algorithm rather than a statistical benchmark.